# บทเรียนที่ 05 - Agentic RAG


## Setup

โน้ตบุ๊กนี้สาธิตรูปแบบ Agentic RAG (Retrieval-Augmented Generation) โดยใช้ Microsoft Agent Framework

**ข้อกำหนดเบื้องต้น:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — จุดเชื่อมต่อบริการ Azure AI Search ของคุณ
- `AZURE_SEARCH_API_KEY` — กุญแจ API สำหรับ Azure AI Search ของคุณ
- การปรับใช้งาน Azure OpenAI ที่กำหนดค่าผ่านตัวแปรแวดล้อม
- การเข้าสู่ระบบ Azure CLI (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAG คืออะไร?

RAG แบบดั้งเดิมจะทำตามขั้นตอนที่กำหนดไว้แล้ว: ดึงเอกสาร แล้วจึงสร้างคำตอบ **Agentic RAG** ไปไกลกว่านั้นด้วยการให้ตัวแทนมีอิสระในการตัดสินใจ **เมื่อใด** และ **อย่างไร** ในการดึงข้อมูล

ด้วย Agentic RAG ตัวแทนสามารถ:
- **ตัดสินใจ** ว่าจำเป็นต้องดึงข้อมูลก่อนตอบคำถามหรือไม่
- **เลือก** แหล่งข้อมูลหรือเครื่องมือที่จะสอบถาม
- **ประเมิน** ผลลัพธ์ที่ดึงมาและดำเนินการดึงข้อมูลซ้ำหากความพยายามครั้งแรกไม่เพียงพอ
- **รวม** ข้อมูลจากหลายขั้นตอนการดึงเข้าด้วยกันเป็นคำตอบที่สอดคล้องกัน

สิ่งนี้ทำให้ตัวแทนมีความยืดหยุ่นและแม่นยำมากกว่ากระบวนการดึงข้อมูลแล้วสร้างคำตอบแบบคงที่


## การสร้างเครื่องมือค้นหา

ใน Agentic RAG แหล่งข้อมูลภายนอกจะถูกห่อหุ้มเป็น **เครื่องมือ** ที่เอเย่นต์สามารถเรียกใช้งานได้ตามต้องการ ซึ่งช่วยให้เอเย่นต์มองว่าการดึงข้อมูลเป็นเพียงการกระทำอีกอย่างหนึ่งที่สามารถทำได้ แทนที่จะเป็นขั้นตอนที่บังคับต้องทำ

ด้านล่างนี้เราจะกำหนดฐานความรู้เกี่ยวกับการเดินทางและเปิดเผยเป็นเครื่องมือที่เอเย่นต์สามารถเรียกใช้เพื่อค้นหาข้อมูลปลายทางได้


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## การสร้างตัวแทน RAG

ตอนนี้เราจะสร้างตัวแทนที่ถูกสั่งให้ **ค้นหาข้อมูลก่อนตอบคำถามเสมอ** ตัวแทนจะใช้เครื่องมือ `search_travel_knowledge` เพื่อยึดคำตอบของตนจากฐานความรู้ แทนที่จะพึ่งพาข้อมูลการฝึกของตนเอง


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## การดึงข้อมูลแบบทำซ้ำ — รูปแบบผู้สร้าง-ผู้ตรวจสอบ

ข้อได้เปรียบที่สำคัญของ Agentic RAG คือ **การดึงข้อมูลแบบทำซ้ำ** ตัวแทนสามารถดำเนินการค้นหาหลายรอบเพื่อยืนยัน ปรับแต่ง หรือขยายผลลัพธ์เบื้องต้น — คล้ายกับกระบวนการทำงาน "ผู้สร้าง-ผู้ตรวจสอบ":

1. **ขั้นตอนผู้สร้าง**: ตัวแทนดึงข้อมูลเบื้องต้นและร่างคำตอบ
2. **ขั้นตอนผู้ตรวจสอบ**: ตัวแทนดำเนินการดึงข้อมูลเพิ่มเติมเพื่อยืนยันรายละเอียดหรือเติมช่องว่าง

ด้านล่างนี้ ตัวแทนถูกถามคำถามที่ต้องเปรียบเทียบปลายทางหลายแห่ง ส่งผลให้ตัวแทนค้นหาหลายครั้ง


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้วิธีสร้างระบบ **Agentic RAG** โดยใช้ Microsoft Agent Framework:

- **Agentic RAG** ช่วยให้เอเจนต์ตัดสินใจดึงข้อมูลด้วยตัวเอง ทำให้การดึงข้อมูลเป็นแบบไดนามิกแทนที่จะกำหนดตายตัว
- **เครื่องมือในฐานะแหล่งข้อมูล**: ฐานความรู้ภายนอก (เช่น Azure AI Search) ถูกห่อหุ้มเป็นเครื่องมือที่เอเจนต์สามารถเรียกใช้ได้
- **การดึงข้อมูลแบบวนซ้ำ**: รูปแบบ maker-checker ช่วยให้เอเจนต์ทำการดึงข้อมูลหลายรอบ — ค้นหา ตรวจสอบ และปรับปรุง — ก่อนที่จะให้คำตอบสุดท้าย

ในสภาพการใช้งานจริง คุณจะต้องแทนที่ `TRAVEL_KNOWLEDGE_BASE` ที่เก็บข้อมูลในหน่วยความจำด้วยดัชนี Azure AI Search ที่แท้จริง เพื่อรองรับการดึงข้อมูลเอกสารการเดินทางในขนาดใหญ่


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ข้อจำกัดความรับผิดชอบ**:  
เอกสารฉบับนี้ได้รับการแปลโดยใช้บริการแปลภาษาอัตโนมัติ [Co-op Translator](https://github.com/Azure/co-op-translator) แม้ว่าเราจะพยายามให้มีความถูกต้องสูงสุด โปรดทราบว่าการแปลอัตโนมัติอาจมีข้อผิดพลาดหรือความคลาดเคลื่อนได้ เอกสารต้นฉบับในภาษาต้นฉบับถือเป็นแหล่งข้อมูลที่มีอำนาจสูงสุด สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้บริการแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความผิดใด ๆ ที่เกิดขึ้นจากการใช้การแปลฉบับนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
